In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, precision_recall_curve, auc, confusion_matrix
import time

In [ ]:
def main():
    print("Loading data...")
    df = pd.read_csv('creditcard.csv')
    
    print("Preprocessing data...")
    # RobustScaler is less prone to outliers, important for 'Amount' and 'Time'
    rob_scaler = RobustScaler()
    df['scaled_amount'] = rob_scaler.fit_transform(df['Amount'].values.reshape(-1,1))
    df['scaled_time'] = rob_scaler.fit_transform(df['Time'].values.reshape(-1,1))
    
    # Drop original columns and reorder
    df.drop(['Time','Amount'], axis=1, inplace=True)
    scaled_amount = df['scaled_amount']
    scaled_time = df['scaled_time']
    df.drop(['scaled_amount', 'scaled_time'], axis=1, inplace=True)
    df.insert(0, 'scaled_amount', scaled_amount)
    df.insert(1, 'scaled_time', scaled_time)
    
    X = df.drop('Class', axis=1)
    y = df['Class']
    
    print("Splitting data...")
    # Stratify is crucial due to class imbalance
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print(f"Original train shape: {X_train.shape}")
    print(f"Class distribution: {y_train.value_counts().to_dict()}")
    
    print("\nApplying SMOTE...")
    start_time = time.time()
    # Apply SMOTE only to training set
    sm = SMOTE(random_state=42)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    print(f"SMOTE finished in {time.time()-start_time:.2f}s")
    print(f"Resampled train shape: {X_train_res.shape}")
    print(f"New class distribution: {y_train_res.value_counts().to_dict()}")
    
    print("\nTraining XGBoost...")
    # hist method is faster on large datasets
    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        eval_metric='aucpr',
        tree_method='hist'
    )
    start_time = time.time()
    model.fit(X_train_res, y_train_res)
    print(f"Training finished in {time.time()-start_time:.2f}s")
    
    print("\nPredicting probabilities on test set...")
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    
    print("Tuning Threshold...")
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_prob)
    
    # Calculate F1 score for each threshold
    f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-10)
    best_threshold_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_threshold_idx]
    best_f1 = f1_scores[best_threshold_idx]
    print(f"Best Threshold: {best_threshold:.4f} with F1-Score: {best_f1:.4f}")
    
    y_pred_tuned = (y_pred_prob >= best_threshold).astype(int)
    
    print("\nClassification Report (Tuned Threshold):")
    print(classification_report(y_test, y_pred_tuned))
    
    print("\nGenerating plots...")
    # 1. Precision-Recall Curve
    plt.figure(figsize=(10, 6))
    plt.plot(recalls, precisions, marker='.', label='XGBoost')
    no_skill = len(y_test[y_test==1]) / len(y_test)
    plt.plot([0, 1], [no_skill, no_skill], linestyle='--', label='No Skill')
    plt.scatter(recalls[best_threshold_idx], precisions[best_threshold_idx], marker='o', color='red', label='Best Threshold', zorder=5)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend()
    plt.grid(True)
    plt.savefig('pr_curve.png')
    plt.close()
    
    # 2. Confusion Matrix
    cm = confusion_matrix(y_test, y_pred_tuned)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix (Threshold={best_threshold:.2f})')
    plt.savefig('confusion_matrix.png')
    plt.close()
    
    # 3. Feature Importance
    fig, ax = plt.subplots(figsize=(10, 8))
    xgb.plot_importance(model, max_num_features=15, ax=ax, importance_type='weight', title='Top 15 Feature Importances (Weight)')
    plt.tight_layout()
    plt.savefig('feature_importance.png')
    plt.close()
    
    print("Done. Saved plots to current directory.")

In [ ]:
if __name__ == "__main__":
    main()